# Week 08B — Object Detection II: YOLO, DETR and Open-Vocabulary Detection

**MCTA 4364 Machine Vision** · Session 2 of Week 8 · 80 minutes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hasanzaki/MCTA-4364-Machine-Vision/blob/main/notebooks/W08_detection/W08B_yolo_detr_open_vocabulary.ipynb) [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/hasanzaki/MCTA-4364-Machine-Vision/blob/main/notebooks/W08_detection/W08B_yolo_detr_open_vocabulary.ipynb)

Session A built detection from boxes, IoU, NMS and AP, and opened a two-stage detector. Today we meet the detectors used in practice in 2025–26:
**one-stage YOLO** models that run in real time on a laptop, **transformer detectors** (DETR, RT-DETR) that remove anchors and NMS, and
**open-vocabulary** detectors that find objects described by text. You will decode a YOLO output tensor by hand, measure speed and accuracy, and see
where zero-shot detection still fails.

### Learning outcomes
By the end of this session you will be able to:
1. Explain dense one-stage prediction (grid, multi-scale heads, anchor-free boxes) and **decode** a raw YOLO output yourself.
2. Pre- and post-process correctly: **letterbox** resizing, confidence filtering, NMS, mapping boxes back to the original image.
3. Measure the **speed–accuracy trade-off** and validate a detector with mAP50 and mAP50-95.
4. Explain **set prediction** with bipartite (Hungarian) matching in DETR and why it needs no NMS.
5. Use an **open-vocabulary** detector with text prompts and judge when zero-shot detection is (not) good enough.

### Session plan (80 min)
| Time | Section | Activity |
|---|---|---|
| 0–10 | 1 | One-stage detection; YOLO architecture |
| 10–30 | 2 | Decoding YOLO by hand: letterbox, 8400 candidates, NMS (interactive) |
| 30–42 | 3 | Speed vs accuracy; validation on coco8 |
| 42–55 | 4 | DETR and RT-DETR: set prediction, Hungarian matching |
| 55–68 | 5–6 | Open-vocabulary detection; detection on video |
| 68–80 | 9 | Quiz and exit ticket |
| Home | 7–8 | Exercises with self-checks, challenge |

**Hardware:** CPU is fine (YOLO11n runs at roughly 5–20 FPS on a laptop CPU and 100+ FPS on a 4 GB GPU). Weights (6–65 MB) download on first use.
The optional open-vocabulary section downloads ~600 MB.

> **Licences.** The `ultralytics` package and its YOLO/RT-DETR weights are **AGPL-3.0**: fine for learning and open-source work, but a commercial
> closed-source product needs an Ultralytics Enterprise licence. Permissive alternatives: torchvision detectors (BSD-3), and RT-DETR, D-FINE,
> RF-DETR or DETR through Hugging Face `transformers` (Apache-2.0).

## 0. Setup

In [ ]:
# Setup: works in Google Colab, Kaggle, Jupyter/VS Code on your laptop, and CI.
import importlib.util
import os
import subprocess
import sys

REPO_URL = "https://github.com/hasanzaki/MCTA-4364-Machine-Vision.git"
if not os.path.exists("resources/scripts/cvhelpers.py"):
    if os.path.exists("../../resources/scripts/cvhelpers.py"):  # opened from its own folder
        os.chdir("../..")
    else:  # Colab / Kaggle: fetch the course helpers
        if not os.path.isdir("MCTA-4364-Machine-Vision"):
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
        os.chdir("MCTA-4364-Machine-Vision")
print("Working directory:", os.getcwd())

# Install only what is missing. PyTorch is deliberately NOT pip-installed here:
# that would replace your laptop's CUDA build with a CPU-only one (see SETUP.md).
needed = {"cv2": "opencv-python", "skimage": "scikit-image", "ipywidgets": "ipywidgets",
          "matplotlib": "matplotlib", "scipy": "scipy", "plotly": "plotly",
          "ultralytics": "ultralytics"}
missing = [pkg for mod, pkg in needed.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
if importlib.util.find_spec("torch") is None:
    raise ImportError("PyTorch is missing. Install it with CUDA support as described in "
                      "SETUP.md, or use Google Colab where it is preinstalled.")
sys.path.append("resources/scripts")

In [ ]:
import time

import cv2
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import torch
import torchvision
from scipy.optimize import linear_sum_assignment
from ultralytics import RTDETR, YOLO

import partsim
from cvhelpers import SMOKE, check, data_dir, interact, load_sample, opencv_sample, pipeline_diagram, quiz, show
from dlhelpers import hw_profile, set_seed

plt.rcParams.update({"figure.dpi": 100, "axes.titlesize": 11})
HW = hw_profile()
DEVICE = HW["device"]
set_seed(0)
WEIGHTS = data_dir("weights")                                  # keep downloaded weights out of the notebook folder
yolo = YOLO(str(WEIGHTS / "yolo11n.pt"))                      # downloaded there on first use
cap = cv2.VideoCapture(str(opencv_sample("vtest.avi")))
_, street = cap.read(); cap.release()
messi = load_sample("messi5.jpg")
scene, sb, sl = partsim.make_scene(4)
print("YOLO11n:", sum(p.numel() for p in yolo.model.parameters()) / 1e6, "M parameters,", len(yolo.names), "COCO classes")

## 1. One-stage detection

A one-stage detector skips the proposal stage: it predicts boxes and classes **densely at every position** of a few feature maps, in one forward pass.

- **YOLO** (Redmon et al., 2016) divided the image into a grid; each cell predicted boxes directly. **SSD** and **RetinaNet** (with the *focal loss*,
  which down-weights the millions of easy background positions) followed.
- Modern YOLO (v8, 11) is **anchor-free**: every location of the stride-8, 16 and 32 feature maps predicts the distances to the four box sides
  (with *distribution focal loss*) and 80 class probabilities. For a 640 × 640 input: $80^2 + 40^2 + 20^2 = 8400$ candidate boxes.
- Architecture: **backbone** (features) → **neck** (FPN + PAN: top-down and bottom-up fusion of scales) → **decoupled head** (separate box and class
  branches) → confidence threshold → **NMS**.
- Training assigns ground-truth objects to good candidate locations (*task-aligned assignment*) and uses IoU-based box losses (CIoU).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(22, 5.5), gridspec_kw=dict(width_ratios=[1.1, 1]))
lb = cv2.resize(messi, (640, 400))
axes[0].imshow(lb[..., ::-1])
for stride, col, lw in [(32, "#ea4335", 1.5), (16, "#34a853", 0.8), (8, "#fbbc05", 0.4)]:
    for x in range(0, 641, stride):
        axes[0].axvline(x, color=col, lw=lw, alpha=0.6)
    for y in range(0, 401, stride):
        axes[0].axhline(y, color=col, lw=lw, alpha=0.6)
axes[0].set_xlim(0, 640); axes[0].set_ylim(400, 0)
axes[0].set_title("three prediction grids: stride 32 (red, large objects), 16 (green), 8 (yellow, small objects)"); axes[0].axis("off")
pipeline_diagram(["640x640\nletterboxed", "backbone\n(C3k2, SPPF)", "neck\nFPN + PAN", "decoupled head\n3 scales", "8400 x (4 box\n+ 80 class)",
                  "threshold\n+ NMS"], colors=["#f1f3f4", "#e8f0fe", "#e6f4ea", "#fef7e0", "#fce8e6", "#f1f3f4"], ax=axes[1], fontsize=9)
axes[1].set_title("YOLO11 inference")
plt.suptitle("Figure 1.1 - Dense one-stage prediction on multi-scale grids", fontweight="bold")
plt.tight_layout(); plt.show()

## 2. Decoding YOLO by hand

`yolo.predict(img)` hides four steps. We do them ourselves:

1. **Letterbox**: resize so the long side is 640 while keeping the aspect ratio, and pad with grey (114) to 640 × 640. Stretching instead would
   distort the objects the network was trained on.
2. **Forward** pass → a tensor of shape `[1, 84, 8400]`: for each of 8400 candidates, $(c_x, c_y, w, h)$ in letterbox pixels + 80 class scores.
3. **Filter**: score = max class probability; keep candidates above the confidence threshold; convert to xyxy; **NMS** per class.
4. **Undo the letterbox**: subtract the padding, divide by the scale, clip to the image.

In [ ]:
def letterbox_ref(img, size=640, color=114):
    h, w = img.shape[:2]
    s = size / max(h, w)
    nh, nw = int(round(h * s)), int(round(w * s))
    px, py = (size - nw) // 2, (size - nh) // 2
    out = np.full((size, size, 3), color, np.uint8)
    out[py:py + nh, px:px + nw] = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_LINEAR)
    return out, s, (px, py)


@torch.no_grad()
def yolo_raw(img):
    lb_img, s, pad = letterbox_ref(img)
    x = torch.tensor(lb_img[..., ::-1].copy()).permute(2, 0, 1)[None].float().div(255)
    yolo.model.eval()
    out = yolo.model.to("cpu")(x)
    out = out[0] if isinstance(out, (list, tuple)) else out
    return out[0].numpy(), lb_img, s, pad                         # [84, 8400]


raw, lb_img, s_, pad_ = yolo_raw(street)
print("raw output:", raw.shape, "-> 4 box numbers + 80 class scores for each of 8400 candidates")
cls_scores = raw[4:]
best = cls_scores.max(0)
fig, axes = plt.subplots(1, 3, figsize=(22, 5.5))
axes[0].imshow(lb_img[..., ::-1]); axes[0].set_title(f"letterboxed input (scale {s_:.3f}, padding {pad_})")
offs = 0
for k, (n, stride) in enumerate([(80, 8), (40, 16), (20, 32)]):
    m = best[offs:offs + n * n].reshape(n, n); offs += n * n
    if k == 1:
        im_ = axes[1].imshow(m, cmap="magma", vmin=0, vmax=1); axes[1].set_title("best class score on the stride-16 grid (40 x 40)")
        plt.colorbar(im_, ax=axes[1], fraction=0.046)
axes[2].hist(best, 100, color="#5f6368"); axes[2].set_yscale("log"); axes[2].axvline(0.25, color="#d93025", ls="--", label="conf = 0.25")
axes[2].set_xlabel("best class score"); axes[2].legend(); axes[2].set_title("almost all 8400 candidates are background")
for ax in axes[:2]:
    ax.axis("off")
plt.suptitle("Figure 2.1 - Inside the YOLO output tensor", fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
def decode_ref(raw, conf=0.25):
    scores = raw[4:].max(0); cls = raw[4:].argmax(0)
    k = scores > conf
    cx, cy, w, h = raw[:4, k]
    return np.stack([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], 1), scores[k], cls[k]


def postprocess(raw, s, pad, shape, conf=0.25, iou=0.7):
    b, sc, cl = decode_ref(raw, conf)
    if len(b):
        keep = torchvision.ops.batched_nms(torch.tensor(b), torch.tensor(sc), torch.tensor(cl), iou).numpy()   # NMS per class
        b, sc, cl = b[keep], sc[keep], cl[keep]
    b = (b - [pad[0], pad[1], pad[0], pad[1]]) / s
    b[:, [0, 2]] = b[:, [0, 2]].clip(0, shape[1]); b[:, [1, 3]] = b[:, [1, 3]].clip(0, shape[0])
    return b, sc, cl


cand, _, _ = decode_ref(raw, 0.25)
ours = postprocess(raw, s_, pad_, street.shape)
ref = yolo.predict(street, conf=0.25, iou=0.7, verbose=False)[0]
names = [yolo.names[int(c)] for c in ours[2]]
fig, axes = plt.subplots(1, 3, figsize=(22, 5.5))
v = lb_img.copy()
for bb in cand:
    cv2.rectangle(v, tuple(bb[:2].astype(int)), tuple(bb[2:].astype(int)), (0, 200, 255), 1)
axes[0].imshow(v[..., ::-1]); axes[0].set_title(f"{len(cand)} candidates above conf 0.25 (before NMS)")
axes[1].imshow(partsim.draw_boxes(street, ours[0], np.arange(len(names)), ours[1], class_names=names)[..., ::-1])
axes[1].set_title(f"our decoding + NMS: {len(ours[0])} objects")
axes[2].imshow(ref.plot()[..., ::-1]); axes[2].set_title(f"yolo.predict(): {len(ref.boxes)} objects")
for ax in axes:
    ax.axis("off")
plt.suptitle("Figure 2.2 - Decoding by hand reproduces the library", fontweight="bold")
plt.tight_layout(); plt.show()

### 🎛️ Interactive: confidence and NMS thresholds
Low confidence finds more (and more wrong) objects; a high NMS IoU threshold keeps overlapping duplicates.

In [ ]:
imgs = {"street (vtest.avi)": street, "messi5.jpg": messi, "fruits.jpg": load_sample("fruits.jpg"), "basketball1.png": load_sample("basketball1.png")}
raws = {k: yolo_raw(v) for k, v in imgs.items()}


def yolo_demo(image="street (vtest.avi)", conf=0.25, nms_iou=0.7):
    r, _, s, p = raws[image]
    b, sc, cl = postprocess(r, s, p, imgs[image].shape, conf, nms_iou)
    show(partsim.draw_boxes(imgs[image], b, np.arange(len(b)), sc, class_names=[yolo.names[int(c)] for c in cl]),
         titles=[f"{image}: conf {conf:.2f}, NMS IoU {nms_iou:.2f} -> {len(b)} detections"], figsize=(10, 7))


interact(yolo_demo, image=list(imgs), conf=widgets.FloatSlider(value=0.25, min=0.02, max=0.9, step=0.01),
         nms_iou=widgets.FloatSlider(value=0.7, min=0.1, max=1.0, step=0.05))

## 3. Speed vs accuracy, and validation

Every detector family comes in sizes (n, s, m, l, x). Published COCO val2017 numbers (Ultralytics documentation, 640 px):

| model | params (M) | mAP50-95 | CPU ONNX (ms) | T4 TensorRT (ms) |
|---|---|---|---|---|
| YOLO11n | 2.6 | 39.5 | 56 | 1.5 |
| YOLO11s | 9.4 | 47.0 | 90 | 2.5 |
| YOLO11m | 20.1 | 51.5 | 183 | 4.7 |
| YOLO11l | 25.3 | 53.4 | 238 | 6.2 |
| YOLO11x | 56.9 | 54.7 | 463 | 11.3 |
| RT-DETR-L | 32 | 53.0 | — | 9.8 (reported) |

Below we measure **your** machine at several input sizes: halving the input size roughly quarters the compute, but small objects vanish.

In [ ]:
sizes = [320, 480, 640] if not SMOKE else [320]
runs = 20 if not SMOKE else 2
res_speed = {}
for sz in sizes:
    yolo.predict(street, imgsz=sz, verbose=False, device=DEVICE)                 # warm-up
    t0 = time.perf_counter()
    for _ in range(runs):
        r = yolo.predict(street, imgsz=sz, verbose=False, device=DEVICE)[0]
    res_speed[sz] = ((time.perf_counter() - t0) / runs * 1000, len(r.boxes))
fig, axes = plt.subplots(1, 2, figsize=(16, 4))
axes[0].bar([str(s) for s in res_speed], [v[0] for v in res_speed.values()], color="#1a73e8")
axes[0].set_xlabel("input size (px)"); axes[0].set_ylabel("ms per image (incl. pre/post-processing)"); axes[0].set_title(f"YOLO11n on {DEVICE}")
axes[1].bar([str(s) for s in res_speed], [v[1] for v in res_speed.values()], color="#188038")
axes[1].set_xlabel("input size (px)"); axes[1].set_ylabel("people and objects found"); axes[1].set_title("detections found (inspect which are real: small, distant people need resolution)")
plt.suptitle("Figure 3.1 - Input size vs speed and detections on your machine", fontweight="bold")
plt.tight_layout(); plt.show()
for k, (ms, n) in res_speed.items():
    print(f"imgsz {k}: {ms:6.1f} ms ({1000 / ms:5.1f} FPS), {n} detections")

**Validation.** `model.val()` runs the detector on a labelled dataset and reports precision, recall, mAP50 and mAP50-95 (the same metrics you
implemented in Session A). `coco8` is a tiny 8-image subset of COCO shipped by Ultralytics (a few MB), useful to check a pipeline, *not* to rank models.

In [ ]:
metrics = yolo.val(data="coco8.yaml", imgsz=640, batch=4, plots=False, verbose=False, device=DEVICE)
print(f"coco8 val: precision {metrics.box.mp:.3f} | recall {metrics.box.mr:.3f} | mAP50 {metrics.box.map50:.3f} | mAP50-95 {metrics.box.map:.3f}")
per_class = {yolo.names[int(c)]: ap for c, ap in zip(metrics.box.ap_class_index, metrics.box.ap)}
plt.figure(figsize=(10, 3.5)); plt.bar(list(per_class), list(per_class.values()), color="#5f6368")
plt.ylabel("AP50-95"); plt.xticks(rotation=30, ha="right"); plt.title("Figure 3.2 - Per-class AP on coco8 (tiny sample: very noisy)", fontweight="bold")
plt.tight_layout(); plt.show()

## 4. Transformer detectors: DETR and RT-DETR

**DETR** (Carion et al., 2020) treats detection as **set prediction**: a transformer decoder turns a fixed number $N$ (e.g. 100 or 300) of learned
**object queries** into $N$ (box, class) predictions, where most predictions are the special class *no object* (∅).

During training, predictions and ground-truth objects are paired by a **bipartite (Hungarian) matching** that minimises a cost combining class
probability, L1 box distance and generalised IoU. Each object is matched to **exactly one** query, so the network learns *not* to produce duplicates:
**no anchors and no NMS**.

**RT-DETR** (Zhao et al., 2024) made DETR real-time with an efficient hybrid encoder and IoU-aware query selection; **D-FINE** and **RF-DETR**
(2024–25) push accuracy further. YOLOv10 and YOLO26 adopted the same idea of NMS-free *one-to-one* assignment.

In [ ]:
# Hungarian matching on a toy example: 6 predictions, 3 ground-truth objects
gt = np.array([[50, 50, 150, 150], [200, 80, 300, 200], [320, 40, 420, 120]], float)
pred = np.array([[55, 48, 148, 155], [60, 60, 160, 160], [205, 90, 295, 210], [310, 50, 410, 130], [100, 200, 180, 260], [210, 70, 310, 190]], float)
p_cls = np.array([0.9, 0.7, 0.8, 0.6, 0.2, 0.75])                   # probability of the correct class


def giou(a, b):
    iou = torchvision.ops.box_iou(torch.tensor(a), torch.tensor(b)).numpy()
    return torchvision.ops.generalized_box_iou(torch.tensor(a), torch.tensor(b)).numpy(), iou


g_, _ = giou(pred, gt)
cost = -p_cls[:, None] + 0.01 * np.abs(pred[:, None] - gt[None]).sum(2) - g_
rows, cols = linear_sum_assignment(cost)
fig, axes = plt.subplots(1, 2, figsize=(18, 4.8))
im_ = axes[0].imshow(cost, cmap="viridis"); plt.colorbar(im_, ax=axes[0], fraction=0.046)
for r, c in zip(rows, cols):
    axes[0].add_patch(plt.Rectangle((c - 0.5, r - 0.5), 1, 1, fill=False, ec="#ea4335", lw=3))
axes[0].set_xlabel("ground-truth object"); axes[0].set_ylabel("prediction (query)"); axes[0].set_title("matching cost; red = Hungarian assignment")
axes[1].set_xlim(0, 450); axes[1].set_ylim(280, 0); axes[1].set_aspect("equal")
for i, gb in enumerate(gt):
    axes[1].add_patch(plt.Rectangle(gb[:2], *(gb[2:] - gb[:2]), fill=False, ec="#188038", lw=3))
for i, pb in enumerate(pred):
    matched = i in rows
    axes[1].add_patch(plt.Rectangle(pb[:2], *(pb[2:] - pb[:2]), fill=False, ec="#1a73e8" if matched else "#9aa0a6", lw=2, ls="-" if matched else "--"))
    axes[1].text(pb[0] + 2, pb[1] + 12, f"q{i}", color="#1a73e8" if matched else "#9aa0a6")
axes[1].set_title("green = GT; blue = matched queries (trained towards the GT); grey = trained to predict 'no object'")
plt.suptitle("Figure 4.1 - DETR's one-to-one matching: duplicates (q1, q5) are pushed to 'no object', so no NMS is needed", fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
rtdetr = RTDETR(str(WEIGHTS / "rtdetr-l.pt"))
res = {}
for name, model in [("YOLO11n (one-stage + NMS)", yolo), ("RT-DETR-L (transformer, no NMS)", rtdetr)]:
    model.predict(street, verbose=False, device=DEVICE)
    t0 = time.perf_counter()
    r = model.predict(street, conf=0.4, verbose=False, device=DEVICE)[0]
    res[name] = (r, (time.perf_counter() - t0) * 1000, sum(p.numel() for p in model.model.parameters()) / 1e6)
fig, axes = plt.subplots(1, 2, figsize=(22, 7))
for ax, (name, (r, ms, npar)) in zip(axes, res.items()):
    ax.imshow(r.plot()[..., ::-1]); ax.axis("off"); ax.set_title(f"{name}: {len(r.boxes)} objects, {npar:.1f} M params, {ms:.0f} ms on {DEVICE}")
plt.suptitle("Figure 4.2 - YOLO11n vs RT-DETR-L on the same frame (conf 0.4)", fontweight="bold")
plt.tight_layout(); plt.show()

## 5. Open-vocabulary detection

A COCO detector can only answer with its 80 classes (Session A: the squirrel became a "bird"). **Open-vocabulary** detectors replace the fixed
classifier by a **similarity with text embeddings**: the class "score" of a region is $\cos(\mathbf{f}_\text{region}, \mathbf{t}_\text{prompt})$, where
$\mathbf{t}$ comes from a CLIP-like text encoder (Week 13). Any noun phrase can become a class at run time.

| model | idea | licence |
|---|---|---|
| OWL-ViT / **OWLv2** (Google) | CLIP ViT with box heads; self-training on web images | Apache-2.0 |
| **Grounding DINO** | DETR + deep text–image fusion | Apache-2.0 |
| **YOLO-World**, **YOLOE** | YOLO with text (and visual) prompts; real time | GPL/AGPL |

The next cell uses **YOLOE** through `ultralytics`. It downloads ~600 MB the first time (a MobileCLIP text encoder), so it is **off by default**:
set `RUN_OPEN_VOCAB = True` to try it on your laptop or in Colab.

In [ ]:
RUN_OPEN_VOCAB = False          # set to True to download YOLOE (~600 MB) and run the open-vocabulary demo

if RUN_OPEN_VOCAB and not SMOKE:
    from ultralytics import YOLOE
    yoloe = YOLOE(str(WEIGHTS / "yoloe-11s-seg.pt"))
    tests = [("messi5.jpg", messi, ["football", "soccer player"]),
             ("squirrel_cls.jpg", load_sample("squirrel_cls.jpg"), ["squirrel", "bird"]),
             ("conveyor scene", scene, ["hex nut", "washer", "bolt", "metal flange"])]
    fig, axes = plt.subplots(1, 3, figsize=(22, 6))
    for ax, (name, img, prompts) in zip(axes, tests):
        yoloe.set_classes(prompts, yoloe.get_text_pe(prompts))
        r = yoloe.predict(img, conf=0.15, verbose=False)[0]
        ax.imshow(r.plot()[..., ::-1]); ax.axis("off"); ax.set_title(f"{name}: prompts {prompts}", fontsize=9)
    plt.suptitle("Figure 5.1 - Zero-shot, text-prompted detection with YOLOE", fontweight="bold")
    plt.tight_layout(); plt.show()
else:
    print("Open-vocabulary demo skipped (RUN_OPEN_VOCAB = False).")
    print("What we observed when preparing this notebook: 'football' is found on messi5.jpg; the squirrel is still called a 'bird';")
    print("on the synthetic conveyor almost every part is called a 'hex nut' with low confidence.")

**Lesson.** Zero-shot detection is superb for *common* objects and a fast way to **bootstrap labels** (Week 13A: auto-labelling), but on specialised
industrial parts it is usually not accurate enough by itself. The dependable route is still: collect and label your own data, then train or fine-tune a
detector (**Week 9**).

## 6. Detection on video: why we need tracking

Running a detector frame by frame gives independent answers. The count of people in `vtest.avi` flickers as people are missed for a frame or two, and
nothing tells us that the person in frame 10 is the same as in frame 11. That is the job of **tracking** (Week 10B).

In [ ]:
cap = cv2.VideoCapture(str(opencv_sample("vtest.avi")))
counts, frames = [], []
n_frames = 12 if SMOKE else 200
for i in range(n_frames):
    ok, fr = cap.read()
    if not ok:
        break
    r = yolo.predict(fr, classes=[0], conf=0.3, verbose=False, device=DEVICE)[0]
    counts.append(len(r.boxes))
    if i in (0, n_frames // 2, n_frames - 1):
        frames.append((i, r.plot()))
cap.release()
fig = plt.figure(figsize=(22, 8))
for k, (i, im_) in enumerate(frames):
    ax = fig.add_subplot(2, 3, k + 1); ax.imshow(im_[..., ::-1]); ax.axis("off"); ax.set_title(f"frame {i}")
ax = fig.add_subplot(2, 1, 2)
ax.plot(counts, "-o", ms=3, color="#1a73e8"); ax.set_xlabel("frame"); ax.set_ylabel("people detected"); ax.grid(alpha=0.3)
ax.set_title("the count flickers from frame to frame: detection alone has no memory")
plt.suptitle("Figure 6.1 - Frame-by-frame detection on vtest.avi", fontweight="bold")
plt.tight_layout(); plt.show()

## 7. Exercises (with self-checks)
Replace each `raise NotImplementedError`, then run the check below it.

### E1. Letterbox
Resize `img` so its longer side equals `size` (keep the aspect ratio), paste it **centred** on a `size × size` canvas filled with 114, and return `(canvas, scale, (pad_x, pad_y))`. Use `round()` for the new size and integer division for the padding.

In [ ]:
def letterbox(img, size=640):
    # TODO: your code here
    raise NotImplementedError

In [ ]:
def _test_e1():
    img = np.zeros((300, 600, 3), np.uint8); img[:, :, 2] = 255
    out, s, (px, py) = letterbox(img, 640)
    assert out.shape == (640, 640, 3), f"expected (640, 640, 3), got {out.shape}"
    assert abs(s - 640 / 600) < 1e-9, f"scale should be 640/600, got {s}"
    assert (px, py) == (0, 160), f"expected padding (0, 160), got {(px, py)}"
    assert out[0, 0, 0] == 114 and out[320, 320, 2] == 255, "padding must be 114 and the image centred"
    ref, s2, p2 = letterbox_ref(street)
    mine, s3, p3 = letterbox(street)
    assert p2 == p3 and np.abs(ref.astype(int) - mine.astype(int)).mean() < 1, "differs from the reference on the street image"


check("E1 letterbox", _test_e1)

### E2. Decode a YOLO output
Given `raw` of shape `[4 + C, N]` (cx, cy, w, h, class scores), return `(boxes_xyxy [K, 4], scores [K], classes [K])` for the candidates whose best class score is **above** `conf`. No NMS.

In [ ]:
def decode_yolo(raw, conf=0.25):
    # TODO: your code here
    raise NotImplementedError

In [ ]:
def _test_e2():
    raw_t = np.zeros((4 + 3, 5), np.float32)
    raw_t[:4, 0] = [100, 50, 20, 10]; raw_t[4 + 2, 0] = 0.9
    raw_t[:4, 3] = [10, 10, 4, 4]; raw_t[4 + 0, 3] = 0.3
    raw_t[4 + 1, 4] = 0.1
    b, s, c = [np.asarray(v) for v in decode_yolo(raw_t, 0.25)]
    assert b.shape == (2, 4), f"expected 2 boxes, got {b.shape}"
    assert np.allclose(b[0], [90, 45, 110, 55]) and c[0] == 2 and abs(s[0] - 0.9) < 1e-6, "first candidate decoded wrongly"
    assert np.allclose(b[1], [8, 8, 12, 12]) and c[1] == 0, "second candidate decoded wrongly"
    b2, s2, c2 = decode_yolo(raw, 0.25)
    b3, s3, c3 = decode_ref(raw, 0.25)
    assert np.allclose(np.asarray(b2), b3) and np.array_equal(np.asarray(c2), c3), "differs from the reference on the real output"


check("E2 decode_yolo", _test_e2)

### E3. Back to the original image
Map letterboxed xyxy boxes back to original-image pixels: subtract the padding, divide by the scale, and **clip** to the image `(height, width)`.

In [ ]:
def unletterbox(boxes, scale, pad, shape):
    # TODO: your code here
    raise NotImplementedError

In [ ]:
def _test_e3():
    b = np.array([[0, 160, 640, 480], [-5, 150, 20, 170]], float)
    out = np.asarray(unletterbox(b, 640 / 600, (0, 160), (300, 600)), float)
    assert np.allclose(out[0], [0, 0, 600, 300]), f"full-image box wrong: {out[0]}"
    assert np.allclose(out[1], [0, 0, 18.75, 9.375]), f"box must be clipped at 0: {out[1]}"
    ours_b = postprocess(raw, s_, pad_, street.shape)[0]
    b2, _, _ = decode_ref(raw, 0.25)
    keep = torchvision.ops.batched_nms(torch.tensor(b2), torch.tensor(decode_ref(raw, 0.25)[1]), torch.tensor(decode_ref(raw, 0.25)[2]), 0.7).numpy()
    assert np.allclose(np.asarray(unletterbox(b2[keep], s_, pad_, street.shape[:2])), ours_b, atol=1e-3), "differs from postprocess() on the street image"


check("E3 unletterbox", _test_e3)

### E4. Your own photos (no self-check)
Run YOLO11n and RT-DETR-L on five photos from your phone (campus, lab, workshop). List every miss and false detection, classify them with the
TIDE categories of Session A (classification, localisation, duplicate, background, missed), and suggest the cheapest fix for each.

In [ ]:
# E4: your code here

## 8. Challenge: a speed–accuracy Pareto front
Using `model.val(data="coco8.yaml")` (or better, `coco128.yaml`, 7 MB), measure mAP50-95 and ms/image for YOLO11n, YOLO11s and RT-DETR-L at input
sizes 320, 480 and 640 on your machine. Plot accuracy vs time and mark the Pareto-optimal points. Which model would you deploy on (a) a laptop CPU at
10 FPS, (b) a 4 GB GPU at 30 FPS? Export the chosen one to ONNX (`model.export(format="onnx")`) and time it with ONNX Runtime (preview of Week 9B).

In [ ]:
# Your challenge code here

## 9. Quick check (auto-marked)

In [ ]:
quiz([
    {
        "q": 'For a 640 x 640 input with strides 8, 16 and 32, how many candidate boxes does YOLO11 output?',
        "options": [
            '640',
            '2100',
            '8400',
            '25200',
        ],
        "answer": '3135bfada1',
        "explain": '80*80 + 40*40 + 20*20 = 8400.',
    },
    {
        "q": 'Why does YOLO letterbox instead of stretching the image to 640 x 640?',
        "options": [
            'It is faster',
            'To keep the object aspect ratio seen in training',
            'To add data augmentation',
            'To remove NMS',
        ],
        "answer": '91c0a3e195',
        "explain": 'Stretching distorts shapes; padding does not.',
    },
    {
        "q": 'What problem does the focal loss (RetinaNet) address?',
        "options": [
            'Small objects',
            'The huge imbalance of easy background vs objects',
            'Slow NMS',
            'Rotated boxes',
        ],
        "answer": '3fb7ba796a',
        "explain": 'It down-weights well-classified (easy) examples.',
    },
    {
        "q": 'Why does DETR not need NMS?',
        "options": [
            'It uses larger anchors',
            'One-to-one Hungarian matching teaches each object to be predicted by one query',
            'It runs at a lower resolution',
            'It predicts masks',
        ],
        "answer": 'd062348a9d',
        "explain": "Duplicates are trained to output 'no object'.",
    },
    {
        "q": 'Halving the input size of a CNN detector changes its compute roughly by a factor of...',
        "options": [
            '1/2',
            '1/4',
            '1/8',
            'no change',
        ],
        "answer": '49c611db64',
        "explain": 'Compute scales with the number of pixels.',
    },
    {
        "q": 'Open-vocabulary detectors classify regions by...',
        "options": [
            'a fixed 80-way softmax',
            'similarity between region and text embeddings',
            'template matching',
            'colour histograms',
        ],
        "answer": '7227dc4438',
        "explain": 'Any text prompt can define a class at run time.',
    },
    {
        "q": 'You must detect a new industrial part with high reliability. Best first step?',
        "options": [
            'Use a zero-shot open-vocabulary detector as the final system',
            'Collect and label images, then fine-tune a detector',
            'Increase the NMS threshold',
            'Use a larger input size only',
        ],
        "answer": '152459ccf0',
        "explain": 'Zero-shot is great for bootstrapping labels, not usually as the final inspection system.',
    },
    {
        "q": 'Under what licence are the ultralytics package and its YOLO weights distributed?',
        "options": [
            'MIT',
            'Apache-2.0',
            'AGPL-3.0',
            'BSD-3',
        ],
        "answer": '6064a58b62',
        "explain": 'Commercial closed-source use needs an enterprise licence; permissive alternatives exist.',
    },
], title='Week 08B quiz')

## 10. Exit ticket (reflection)
1. Draw the path of one pixel of a detected person from the camera frame to the final box, naming every coordinate transform.
2. When would you choose RT-DETR over YOLO, and the reverse?
3. Name one task in your project where an open-vocabulary detector could save labelling time, and how you would check its output.

## 11. What's new (2025–26)

> - **NMS-free real-time detectors**: YOLOv10, YOLO26 and the DETR family (RT-DETRv2, D-FINE, DEIM, RF-DETR) use one-to-one assignment, which
>   simplifies deployment and makes latency predictable.
> - **Open-set and prompted detection**: YOLOE and YOLO-World (text and visual prompts), Grounding DINO 1.5, OWLv2 and Florence-2 detect arbitrary
>   phrases; SAM 2/3-style models add masks (Week 11B). Vision-language models such as Qwen2.5-VL output boxes from chat prompts (Week 13A).
> - **Benchmarks for the real world**: RF100-VL and ODinW measure how well detectors transfer to specialised domains (aerial, medical, industrial),
>   where COCO mAP is a poor predictor.

## 12. References and further exploration

**Seminal papers**
- Redmon, J. et al. (2016). *You only look once: Unified, real-time object detection.* CVPR. [arXiv:1506.02640](https://arxiv.org/abs/1506.02640)
- Liu, W. et al. (2016). *SSD: Single shot multibox detector.* ECCV. [arXiv:1512.02325](https://arxiv.org/abs/1512.02325)
- Lin, T.-Y. et al. (2017). *Focal loss for dense object detection* (RetinaNet). ICCV. [arXiv:1708.02002](https://arxiv.org/abs/1708.02002)
- Tian, Z. et al. (2019). *FCOS: Fully convolutional one-stage object detection* (anchor-free). ICCV. [arXiv:1904.01355](https://arxiv.org/abs/1904.01355)
- Carion, N. et al. (2020). *End-to-end object detection with transformers* (DETR). ECCV. [arXiv:2005.12872](https://arxiv.org/abs/2005.12872)
- Zhao, Y. et al. (2024). *DETRs beat YOLOs on real-time object detection* (RT-DETR). CVPR. [arXiv:2304.08069](https://arxiv.org/abs/2304.08069)
- Wang, A. et al. (2024). *YOLOv10: Real-time end-to-end object detection.* NeurIPS. [arXiv:2405.14458](https://arxiv.org/abs/2405.14458)
- Minderer, M. et al. (2023). *Scaling open-vocabulary object detection* (OWLv2). NeurIPS. [arXiv:2306.09683](https://arxiv.org/abs/2306.09683)
- Liu, S. et al. (2024). *Grounding DINO.* ECCV. [arXiv:2303.05499](https://arxiv.org/abs/2303.05499)
- Cheng, T. et al. (2024). *YOLO-World: Real-time open-vocabulary object detection.* CVPR. [arXiv:2401.17270](https://arxiv.org/abs/2401.17270)
- Wang, A. et al. (2025). *YOLOE: Real-time seeing anything.* [arXiv:2503.07465](https://arxiv.org/abs/2503.07465)

**Documentation and tools**
- Ultralytics docs: *Predict*, *Val*, *Models* (YOLO11, RT-DETR, YOLOE) and the licence page. [docs.ultralytics.com](https://docs.ultralytics.com/)
- Hugging Face `transformers` object-detection guide (DETR, RT-DETR, OWLv2, Grounding DINO; Apache-2.0). [huggingface.co/docs/transformers](https://huggingface.co/docs/transformers/tasks/object_detection)
- Roboflow *Model Leaderboard* and RF100-VL benchmark. [leaderboard.roboflow.com](https://leaderboard.roboflow.com/)

**Images:** `vtest.avi`, `messi5.jpg`, `fruits.jpg`, `basketball1.png`, `squirrel_cls.jpg` from the OpenCV sample data; `coco8` from Ultralytics (COCO, CC BY 4.0).

## 13. Key takeaways
- One-stage detectors predict **densely** on multi-scale grids (8400 candidates for YOLO11 at 640 px) and filter with a threshold and **NMS**.
- Correct **pre/post-processing** (letterbox, decoding, un-letterbox) is part of the model; get it wrong and accuracy collapses.
- Choose model size and input size from a **measured** speed–accuracy trade-off on your hardware.
- **DETR/RT-DETR**: object queries + Hungarian matching = set prediction without anchors or NMS.
- **Open-vocabulary** detectors are flexible and useful for bootstrapping, but specialised parts still need your own data (Week 9).

**Next week (09A):** data-centric machine vision — the data matter more than the model.